In [ ]:
# Imports

import sys
from pathlib import Path
import os
import yaml
import echopype as ep
from aa_si_visualization import echogram

sys.path.insert(0, str(Path("../src").resolve()))
from aa_si_calibration.calibration import generate_standardized_cal_mapping

%load_ext autoreload
%autoreload 2

from aa_si_calibration import calibration
from aa_si_calibration import comparison
from aa_si_ml import ml
from aa_si_utils import utils
from aa_si_utils.data_retrieval import query_ncei_data, download_ncei_data

print("All imports successful!")


In [ ]:
# Load recipe

RECIPE_PATH = "./simplified_workshop_recipe.yaml"

with open(RECIPE_PATH, "r") as f:
    config = yaml.safe_load(f)

# Extract top-level config sections
env = config["local_environment"]
cal_cfg = config["calibration"]["calibration_pipeline"]
masking_cfg = config["masking"]
noise_cfg = config["noise_removal"]
mvbs_cfg = config["mvbs"]
line_cfg = config["line_files"]
echogram_cfg = config["echogram_visualization"]
ml_cfg = config["ml"]
retrieval_cfg = config["data_retrieval"]

setup = env["initial_setup"]
raw_combine_cfg = env["open_and_combine_raw_files"]

global_params = {
    "cruise_id": cal_cfg["cruise_id"],
    "record_author": cal_cfg["record_author"],
}

# Build overlay line definitions for echogram visualization
overlay_lines = [
    {
        "var": f'{line_cfg["line_name"]}{line["var_suffix"]}',
        "style": line["style"],
    }
    for line in line_cfg["overlay_lines"]
]

print(f"Recipe loaded from: {RECIPE_PATH}")
print(f"Record author:  {cal_cfg['record_author']}")
print(f"Cruise ID:      {cal_cfg['cruise_id']}")
print(f"\nInput folders:")
print(f"  Raw files:         {setup['raw_input_folder']}")
print(f"  Calibration files: {setup['cal_input_folder']}")
print(f"\nOutput base: {setup['output_base']}")


In [ ]:
# Query NCEI for available files

results = query_ncei_data(
    file_time_start=retrieval_cfg["file_time_start"],
    file_time_end=retrieval_cfg["file_time_end"],
    filters=retrieval_cfg["filters"],
)

for r in results:
    print(f"  {r['FILE_NAME']}  ({r['FILE_DATETIME']})")


In [ ]:
# Download raw files

downloaded_paths = download_ncei_data(
    results,
    output_dir=setup["raw_input_folder"],
    companion_extensions=retrieval_cfg["companion_extensions"],
)


In [ ]:
# Validate local files

raw_file_paths = utils.initial_setup_and_validation(
    raw_input_folder=setup["raw_input_folder"],
    netcdf_output_folder_string=setup["netcdf_output_folder"],
    sv_output_folder_string=setup["sv_output_folder"],
    output_logs_folder_string=setup["output_logs_folder"],
    raw_file_names=setup["raw_file_names"],
    clear_previous_json_logs=setup["clear_previous_json_logs"],
)


In [ ]:
# Generate Standardized Calibration Mapping
#   - Read raw file configurations
#   - Convert manufacturer calibration files to single-channel standardized files
#   - Match raw channels to calibration data and save the mapping

pipeline_result = generate_standardized_cal_mapping(
    raw_input_folder=setup["raw_input_folder"],
    cal_input_folder=setup["cal_input_folder"],
    output_base=setup["output_base"],
    global_params=global_params,
    short_filenames=cal_cfg["short_filenames"],
    keep_unused=cal_cfg["keep_unused_standardized_files"],
    conflict_resolution=cal_cfg["conflict_resolution"],
)

mapping_dict = pipeline_result["mapping_dict"]
calibration_dict = pipeline_result["calibration_dict"]


In [ ]:
# Load raw data

echodata = utils.open_and_combine_raw_files(
    raw_file_paths,
    setup["netcdf_output_folder"],
    sonar_model=raw_combine_cfg["sonar_model"],
    include_bot=raw_combine_cfg["include_bot"],
)


In [ ]:
# Extract calibration parameters from raw/NetCDF file

# original_params = calibration.extract_netcdf_calibration_parameters(echodata, setup["output_logs_folder"])

# nc_cal_files = [os.path.basename(raw_file_paths[0])]
# original_params["other_params"]["source_filenames_across_channels"] = nc_cal_files
# original_params["other_params"]["source_file_type"] = ".raw"

# calibration.print_calibration_values(echodata, original_params, "Calibration Values From .nc File")


In [ ]:
# Extract calibration parameters from standardized files

standardized_params = calibration.extract_standardized_calibration_parameters(
    calibration_dict, mapping_dict, echodata=echodata,
)

standardized_params["other_params"]["source_filenames_across_channels"] = [os.path.basename(raw_file_paths[0])]
standardized_params["other_params"]["source_file_type"] = ".cal"

calibration.print_calibration_values(echodata, standardized_params, "Calibration Values From Standardized Files")


In [ ]:
# comparison.compare_calibration_parameters(standardized_params, original_params, echodata)


In [ ]:
# Compute Sv datasets and create mask

ds_Sv_baseline_unmasked = ep.calibrate.compute_Sv(echodata)

cal_params_combined = {
    "gain_correction": standardized_params["cal_params"]["gain_correction"],
    "sa_correction": standardized_params["cal_params"]["sa_correction"],
    "equivalent_beam_angle": standardized_params["cal_params"]["equivalent_beam_angle"],
}

env_params_combined = {
    "sound_speed": standardized_params["env_params"]["sound_speed"],
    "sound_absorption": standardized_params["env_params"]["sound_absorption"],
}

ds_Sv_calibrated_unmasked = ep.calibrate.compute_Sv(
    echodata, cal_params=cal_params_combined, env_params=env_params_combined
)

seafloor_params = masking_cfg["remove_seafloor_from_mask"]
surface_params = masking_cfg["remove_surface_from_mask"]
freq_params = masking_cfg["mask_frequency_channels"]

mask = utils.createSvMask(ds_Sv_baseline_unmasked)
mask = utils.remove_seafloor_from_mask(
    echodata,
    ds_Sv_baseline_unmasked,
    mask,
    buffer_m=seafloor_params["seafloor_buffer_m"],
    use_best_detection=seafloor_params["use_best_detection"],
    min_valid_depth_m=seafloor_params["min_valid_depth_m"],
)
mask = utils.remove_surface_from_mask(
    ds_Sv_baseline_unmasked,
    mask,
    depth_threshold_m=surface_params["surface_depth_m"],
)
mask = utils.mask_frequency_channels(
    ds_Sv_baseline_unmasked,
    mask,
    freq_params["frequencies_to_mask"],
)
utils.log_mask_stats(mask)


In [ ]:
# sv_datasets = comparison.compute_calibrated_sv_datasets(echodata, standardized_params)


In [ ]:
# Apply mask and save baseline

ds_Sv_baseline = utils.apply_mask_to_sv(ds_Sv_baseline_unmasked, mask)
# calibrated_sv = utils.apply_mask_to_sv_datasets(sv_datasets, mask)
ds_Sv_calibrated = utils.apply_mask_to_sv(ds_Sv_calibrated_unmasked, mask)

ds_Sv_baseline.to_netcdf(setup["sv_output_folder"] + "/NEFSC_processed_data.nc")
print("Mask applied and baseline saved")


In [ ]:
# comparison_results = comparison.run_sv_comparison_analysis(
#     ds_Sv_baseline,
#     calibrated_sv,
#     echodata,
#     original_params,
#     setup["output_logs_folder"],
#     sv_flag_thresholds=sv_comp_cfg["sv_flag_thresholds"],
# )

# ds_Sv_calibrated = calibrated_sv["combined"]
# verification_results = comparison_results["verification_results"]


In [ ]:
# Noise removal, MVBS, and echogram plotting

noise_params = noise_cfg["remove_noise"]
ds_Sv_clean = ml.remove_noise(
    ds_Sv_calibrated,
    noise_range_sample_num=noise_params["noise_range_sample_num"],
    noise_ping_num=noise_params["noise_ping_num"],
)

# Mask sparse bins before computing MVBS
sparse_params = mvbs_cfg["mask_sparse_bins"]
ds_Sv_clean = utils.mask_sparse_bins(
    ds_Sv_clean,
    range_bin=sparse_params["range_bin"],
    ping_time_bin=sparse_params["ping_time_bin"],
    nan_threshold=sparse_params["nan_threshold"],
)

# Compute Mean Volume Backscattering Strength (MVBS)
mvbs_params = mvbs_cfg["compute_MVBS"]
ds_MVBS = ep.commongrid.compute_MVBS(
    ds_Sv_clean,
    range_bin=mvbs_params["range_bin"],
    ping_time_bin=mvbs_params["ping_time_bin"],
)

# Add dive profile overlay data
ds_MVBS = utils.add_dive_profile_to_dataset(
    ds_MVBS,
    line_cfg["line_file_path"],
    line_cfg["line_name"],
)

# Plot echograms
echo_params = echogram_cfg["plot_sv_echogram"]

echogram.plot_sv_echogram(
    ds_Sv_clean,
    min_depth=echo_params["min_depth"],
    max_depth=echo_params["max_depth"],
    ping_min=echo_params["ping_min"],
    ping_max=echo_params["ping_max"],
    x_axis_units=echo_params["x_axis_units"],
    y_axis_units=echo_params["y_axis_units"],
    echodata=echodata,
)

echogram.plot_sv_echogram(
    ds_MVBS,
    ds_Sv_clean,
    min_depth=echo_params["min_depth"],
    max_depth=echo_params["max_depth"],
    ping_min=echo_params["ping_min"],
    ping_max=echo_params["ping_max"],
    x_axis_units=echo_params["x_axis_units"],
    y_axis_units=echo_params["y_axis_units"],
    echodata=echodata,
    overlay_lines=overlay_lines,
)


In [ ]:
# Reshape data for ML

reshape_cfg = ml_cfg["reshape_data_for_ml"]
norm_cfg = ml_cfg["normalize_data"]
hdbscan_cfg = ml_cfg["hdbscan"]
aux_cfg = ml_cfg["add_auxiliary_features"]

ds_ml_ready = ml.reshape_data_for_ml(
    ds_MVBS,
    data_var=reshape_cfg["data_var"],
    dataset_name=reshape_cfg["dataset_name"],
    feature_strategy=reshape_cfg["feature_strategy"],
    baseline_channel=reshape_cfg["baseline_channel"],
)


In [ ]:
# Add depth as an auxiliary feature
ds_ml_ready = ml.add_auxiliary_features(
    ds_ml_ready,
    dataset_name=reshape_cfg["dataset_name"],
    features=aux_cfg["features"]
)


In [ ]:
# Normalize data and plot echogram

ds_normalized = ml.normalize_data(
    ds_ml_ready,
    method=norm_cfg["method"],
    shift_positive=norm_cfg["shift_positive"],
    dataset_name=reshape_cfg["dataset_name"],
    normalization_name=norm_cfg["normalization_name"],
    feature_weights=norm_cfg["feature_weights"],
    per_group_methods=norm_cfg["per_group_methods"]
)

echogram.plot_flattened_data_echogram(
    ds_normalized,
    reshape_cfg["dataset_name"],
    ds_Sv_clean,
    ml_specific_data_name=norm_cfg["normalization_name"],
    min_depth=echo_params["min_depth"],
    max_depth=echo_params["max_depth"],
    ping_min=echo_params["ping_min"],
    ping_max=echo_params["ping_max"],
    x_axis_units=echo_params["x_axis_units"],
    y_axis_units=echo_params["y_axis_units"],
)


In [ ]:
# Run HDBSCAN clustering
ds_MVBS, gridded_background_results_dbscan, dbscan_results = ml.extract_data_and_run_hdbscan(
    ds_normalized,
    reshape_cfg["dataset_name"],
    ds_Sv_original=ds_Sv_clean,
    custom_normalization_name=norm_cfg["normalization_name"],
    ml_result_name=hdbscan_cfg["ml_result_name"],
    plot_window=hdbscan_cfg["plot_window"],
    min_samples=hdbscan_cfg["min_samples"],
    sample_size=hdbscan_cfg["sample_size"],
    min_cluster_size=hdbscan_cfg["min_cluster_size"],
    cluster_selection_method="leaf", #hdbscan_cfg["cluster_selection_method"],
    use_hdbscan=hdbscan_cfg["use_hdbscan"],
    cluster_colors=ml_cfg["cluster_colors"],
    overlay_line_var=line_cfg["line_name"],
)